# Stage 3 XGBoost IDS Training

This notebook is a Stage 3 template for training a future XGBoost IDS classifier using CSE-CIC-IDS2018 flow features.

Large raw CSV files should not be committed to GitHub. Use Google Colab, local ignored folders, or external storage for raw dataset files.

Feature-label separation must be preserved: labels are allowed for supervised training and evaluation, but prediction must use feature-only records.

## Install / Import Dependencies

In [ ]:
# In Colab, uncomment if needed:
# !pip install pandas scikit-learn xgboost

from pathlib import Path
import json

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

RANDOM_STATE = 42

## Load CSE-CIC-IDS2018 Data

In [ ]:
# Option A: load Stage 1 processed feature sample and ground truth.
# This searches from the current working directory so the notebook can run
# from the repo root or from stage-3/notebooks.
candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = next(root for root in candidate_roots if (root / 'stage-1').exists())
FEATURE_SAMPLE_PATH = REPO_ROOT / 'stage-1' / 'data' / 'processed' / 'flow-feature-sample.csv'
GROUND_TRUTH_PATH = REPO_ROOT / 'stage-1' / 'data' / 'processed' / 'ground-truth.json'

# Option B: replace this with a Colab/KaggleHub/raw CSV path for full training later.
# RAW_CSV_PATH = Path('/content/path/to/cse-cic-ids2018.csv')

features_df = pd.read_csv(FEATURE_SAMPLE_PATH)
with open(GROUND_TRUTH_PATH, 'r', encoding='utf-8') as file:
    ground_truth = json.load(file)

features_df.head()

## Clean and Validate Data

In [ ]:
FORBIDDEN_FEATURE_FIELDS = {
    'Label', 'rawLabel', 'attackType', 'mappedAttackType',
    'groundTruth', 'severity', 'similarityKey'
}

forbidden_present = sorted(FORBIDDEN_FEATURE_FIELDS.intersection(features_df.columns))
if forbidden_present:
    raise ValueError(f'Forbidden answer fields found in feature input: {forbidden_present}')

features_df = features_df.replace([float('inf'), float('-inf')], pd.NA)
features_df = features_df.dropna()
features_df.shape

## Map Raw Labels to Dashboard Attack Types

In [ ]:
LABEL_MAPPING = {
    'Benign': 'Benign',
    'FTP-BruteForce': 'Brute Force',
    'SSH-Bruteforce': 'Brute Force',
    'Bot': 'Botnet',
    'DoS attacks-GoldenEye': 'DoS',
    'DoS attacks-Slowloris': 'DoS',
    'DoS attacks-SlowHTTPTest': 'DoS',
    'DoS attacks-Hulk': 'DoS',
    'DDoS attacks-LOIC-HTTP': 'DDoS',
    'DDOS attack-HOIC': 'DDoS',
    'DDOS attack-LOIC-UDP': 'DDoS',
    'Brute Force -Web': 'Web Attack',
    'Brute Force -XSS': 'Web Attack',
    'SQL Injection': 'Web Attack',
    # CSE-CIC-IDS2018 sources may use spelling variants; handle both.
    'Infilteration': 'Infiltration',
    'Infiltration': 'Infiltration',
}

labels = features_df['id'].map(lambda alert_id: ground_truth.get(alert_id, {}).get('mappedAttackType'))
labels.value_counts(dropna=False)

## Select Feature Columns

In [ ]:
feature_columns = [column for column in features_df.columns if column != 'id']

# Encode categorical flow fields such as protocol before training.
X = pd.get_dummies(features_df[feature_columns], columns=['protocol'], drop_first=False)
y = labels

X.head(), y.head()

## Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train.shape, X_test.shape

## Train XGBoost Classifier

In [ ]:
# Scaffold only. Final hyperparameters should be chosen later using a proper validation split.
# model = XGBClassifier(
#     n_estimators=200,
#     max_depth=6,
#     learning_rate=0.1,
#     objective='multi:softprob',
#     eval_metric='mlogloss',
#     random_state=RANDOM_STATE,
# )
# model.fit(X_train, y_train)

print('Training scaffold prepared. Final model training is not run by default.')

## Evaluate Model

In [ ]:
# After training:
# y_pred = model.predict(X_test)
# print(classification_report(y_test, y_pred))
# confusion_matrix(y_test, y_pred)

print('Evaluation scaffold: accuracy, precision, recall, F1, confusion matrix, false positives, and false negatives.')

## Export Model Artifacts

In [ ]:
MODEL_DIR = REPO_ROOT / 'stage-3' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Future artifacts:
# model.save_model(MODEL_DIR / 'xgboost_ids_model.json')
# (MODEL_DIR / 'feature-columns.json').write_text(json.dumps(list(X.columns), indent=2))
# (MODEL_DIR / 'label-mapping.json').write_text(json.dumps(LABEL_MAPPING, indent=2))
# (MODEL_DIR / 'preprocessing-config.json').write_text(json.dumps({'forbiddenFeatureFields': sorted(FORBIDDEN_FEATURE_FIELDS)}, indent=2))

print('Artifact export scaffold prepared.')

## Generate Sample ML Predictions

In [ ]:
# Future output format:
# predictions = [
#     {
#         'id': row_id,
#         'predictedAttackType': predicted_label,
#         'modelConfidence': float(confidence),
#         'baseRiskScore': int(round(confidence * 100)),
#     }
# ]
# Path('../outputs/ml-predictions.sample.json').write_text(json.dumps(predictions, indent=2))

print('Prediction export scaffold prepared. Do not use ground truth to create predictions.')

## Notes and Limitations

- This notebook is a scaffold, not a final trained model.
- Raw CSE-CIC-IDS2018 CSV files should remain outside GitHub.
- Prediction must use feature-only input.
- Ground truth can be joined only after prediction for evaluation.
- Future evaluation should report accuracy, precision, recall, F1, confusion matrix, per-class performance, false positives, and false negatives.